# Choob — Optimized Existing CLIP Recognition

This reproduces prompt, vocabulary and scope-check improvements to the original CLIP ViT-B/32 weights. No weights are fine-tuned. Start with a fresh **CPU Colab runtime**, then run the cells in order.

Recorded run on 23 October 2026: 260 calibration + 340 holdout photos; supported Top-1 175/180 → 176/180; unsupported photos marked uncertain 0/160 → 159/160. The optimized system also marked 16/180 supported photos uncertain. These are recorded results; rerunning the cells below creates a fresh runtime output. See the bundled optimization report for limits.


In [ ]:
%pip -q install "transformers==4.57.6" pillow requests flask


In [ ]:
%%writefile recognition.py
"""Choob CLIP inference: cached prompt ensembles and explicit scope checking.

Uses the original pretrained weights; no weight fine-tuning. Supported scores
rank nine dish families. A separate, broader vocabulary competes with those
families so an unsupported image need not be forced into the app's menu.
"""
from __future__ import annotations

from hashlib import sha256
import json
import math
from pathlib import Path

MODEL_ID = "openai/clip-vit-base-patch32"
VERSION = "clip-scope-v2"
BASELINE_LABELS = [
    "tonkotsu ramen", "shoyu ramen", "sushi", "dumplings", "fried rice",
    "pad thai", "pizza", "pasta", "hamburger", "tacos",
]
TEMPLATES = [
    "a photo of {}.", "a close-up photo of {}.",
    "a photo of a plate of {}.", "a restaurant photo of {}.",
]
# Ramen is intentionally a family label: the available evaluation labels do
# not establish reliable tonkotsu versus shoyu recognition.
SUPPORTED_DESCRIPTIONS = {
    "ramen": ["ramen", "Japanese ramen noodle soup", "tonkotsu ramen", "shoyu ramen"],
    "sushi": ["sushi", "Japanese sushi rolls", "nigiri sushi"],
    "dumplings": ["dumplings", "gyoza", "potstickers", "steamed Asian dumplings"],
    "fried rice": ["fried rice", "Chinese fried rice", "egg fried rice"],
    "pad thai": ["pad thai", "Thai stir fried rice noodles"],
    "pizza": ["pizza", "pepperoni pizza", "cheese pizza"],
    "pasta": ["pasta", "spaghetti bolognese", "spaghetti carbonara", "Italian pasta with sauce", "lasagna", "ravioli", "gnocchi", "macaroni and cheese"],
    "hamburger": ["hamburger", "cheeseburger", "beef burger in a bun"],
    "tacos": ["tacos", "Mexican tacos"],
}
# Fixed before the new calibration/test run. Food names use Food-101's public
# taxonomy, plus other foods and explicit non-food descriptions. This is a
# competing vocabulary, not a guarantee of open-world rejection.
BACKGROUND_DESCRIPTIONS = [
    "apple pie", "baby back ribs", "baklava", "beef carpaccio", "beef tartare",
    "beet salad", "beignets", "bibimbap", "bread pudding", "breakfast burrito",
    "bruschetta", "caesar salad", "cannoli", "caprese salad", "carrot cake",
    "ceviche", "cheesecake", "cheese plate", "chicken curry", "chicken quesadilla",
    "chicken wings", "chocolate cake", "chocolate mousse", "churros", "clam chowder",
    "club sandwich", "crab cakes", "creme brulee", "croque madame", "cup cakes",
    "deviled eggs", "donuts", "edamame", "eggs benedict", "escargots", "falafel",
    "filet mignon", "fish and chips", "foie gras", "french fries", "french onion soup",
    "french toast", "fried calamari", "frozen yogurt", "garlic bread", "Greek salad",
    "grilled cheese sandwich", "grilled salmon", "guacamole", "gyros", "hot and sour soup",
    "hot dog", "huevos rancheros", "hummus", "ice cream", "lobster bisque",
    "lobster roll sandwich", "macarons", "miso soup", "mussels", "nachos",
    "omelette", "onion rings", "oysters", "paella", "pancakes", "panna cotta",
    "Peking duck", "pho noodle soup", "pork chop", "poutine", "prime rib",
    "pulled pork sandwich", "red velvet cake", "risotto", "samosa", "sashimi",
    "scallops", "seaweed salad", "shrimp and grits", "spring rolls", "steak",
    "strawberry shortcake", "takoyaki", "tiramisu", "tuna tartare", "waffles",
    "fresh fruit", "a vegetable salad", "a slice of cake", "a bowl of soup",
    "a cup of coffee", "a glass of juice", "a bottle of water",
    "a dog", "a cat", "a person", "a building", "a landscape", "a car",
    "a mobile phone", "a laptop computer", "a book", "a shoe", "an empty plate",
    "a screenshot with text", "a non-food object",
]
CATEGORY_MAP = {
    "ramen": ("Japanese", "Noodles"), "sushi": ("Japanese", "Seafood"),
    "dumplings": ("Chinese / Asian", "Dumplings"), "fried rice": ("Asian", "Rice Dishes"),
    "pad thai": ("Thai", "Noodles"), "pizza": ("Italian", "Pizza"),
    "pasta": ("Italian", "Pasta"), "hamburger": ("American", "Sandwiches & Burgers"),
    "tacos": ("Mexican", "Other"),
}


def vocabulary_fingerprint():
    spec = {"version": VERSION, "model": MODEL_ID, "templates": TEMPLATES,
            "supported": SUPPORTED_DESCRIPTIONS, "background": BACKGROUND_DESCRIPTIONS}
    return sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()


def load_threshold(path=None):
    """A saved threshold is valid only for this exact prompt vocabulary."""
    path = Path(path) if path else Path(__file__).with_name("recognition_config.json")
    config = json.loads(path.read_text(encoding="utf-8"))
    if config.get("vocabulary_fingerprint") != vocabulary_fingerprint():
        raise ValueError("Recognition vocabulary changed; recalibrate before serving.")
    threshold = float(config["support_margin_threshold"])
    if not math.isfinite(threshold) or not -2 <= threshold <= 2:
        raise ValueError("Invalid recognition threshold")
    return threshold


class DishRecognizer:
    def __init__(self, model, processor, threshold=0.0):
        import torch
        import torch.nn.functional as F
        self.model, self.processor, self.threshold = model, processor, float(threshold)
        self.device = next(model.parameters()).device
        self.labels = list(SUPPORTED_DESCRIPTIONS)
        self.model.eval()
        self.groups = []
        descriptions = []
        for label in self.labels:
            first = len(descriptions)
            descriptions.extend(SUPPORTED_DESCRIPTIONS[label])
            self.groups.append((first, len(descriptions)))
        self.support_count = len(descriptions)
        descriptions.extend(BACKGROUND_DESCRIPTIONS)
        prompts = [template.format(name) for name in descriptions for template in TEMPLATES]
        text = self._encode_text(prompts)
        self.prototypes = F.normalize(text.reshape(len(descriptions), len(TEMPLATES), -1).mean(1), dim=-1)
        self.baseline_text = self._encode_text([f"a photo of {name}" for name in BASELINE_LABELS])
        self.logit_scale = float(model.logit_scale.detach().exp().cpu())

    def _encode_text(self, prompts, batch_size=64):
        import torch
        import torch.nn.functional as F
        rows = []
        with torch.inference_mode():
            for start in range(0, len(prompts), batch_size):
                inputs = self.processor(text=prompts[start:start + batch_size], padding=True, return_tensors="pt")
                inputs = {key: value.to(self.device) for key, value in inputs.items()}
                features = self.model.get_text_features(**inputs)
                rows.append(F.normalize(features, dim=-1).cpu())
        return torch.cat(rows)

    def encode_images(self, images, batch_size=16):
        import torch
        import torch.nn.functional as F
        rows = []
        with torch.inference_mode():
            for start in range(0, len(images), batch_size):
                inputs = self.processor(images=images[start:start + batch_size], return_tensors="pt")
                pixels = inputs["pixel_values"].to(self.device)
                features = self.model.get_image_features(pixel_values=pixels)
                rows.append(F.normalize(features, dim=-1).cpu())
        return torch.cat(rows)

    def score_features(self, features):
        import torch
        similarities = features.cpu() @ self.prototypes.T
        supported = torch.stack([similarities[:, start:end].max(dim=1).values for start, end in self.groups], dim=1)
        relative = (supported * self.logit_scale).softmax(dim=1)
        background, background_idx = similarities[:, self.support_count:].max(dim=1)
        rows = []
        for i in range(len(features)):
            order = supported[i].argsort(descending=True).tolist()
            best = float(supported[i, order[0]])
            rows.append({
                "top_3": [{"dish_name": self.labels[j], "relative_score": float(relative[i, j])} for j in order[:3]],
                "support_margin": best - float(background[i]),
                "support_similarity": best, "background_similarity": float(background[i]),
                "background_label": BACKGROUND_DESCRIPTIONS[int(background_idx[i])],
            })
        return rows

    def baseline_scores(self, features):
        scores = (features.cpu() @ self.baseline_text.T * self.logit_scale).softmax(dim=1)
        return [{"top_3": [{"dish_name": BASELINE_LABELS[j], "relative_score": float(row[j])}
                            for j in row.argsort(descending=True).tolist()[:3]]} for row in scores]

    def predict_from_scores(self, score, threshold=None):
        limit = self.threshold if threshold is None else float(threshold)
        accepted = score["support_margin"] >= limit
        suggestion = score["top_3"][0]["dish_name"]
        cuisine, category = CATEGORY_MAP[suggestion] if accepted else ("Unknown", "Unknown")
        return {
            "dish_name": suggestion.title() if accepted else "Unknown",
            "suggested_dish": suggestion.title(),
            "cuisine": cuisine, "category": category,
            "status": "supported" if accepted else "uncertain", "accepted": bool(accepted),
            "relative_score": round(score["top_3"][0]["relative_score"], 5),
            "alternatives": [{"dish_name": row["dish_name"].title(), "relative_score": round(row["relative_score"], 5)} for row in score["top_3"][1:]],
            "needs_user_confirmation": True, "supported_labels": self.labels,
            "model": "CLIP ViT-B/32 · prompt ensemble + scope check",
            "score_note": "Relative ranking among nine supported dish families; not calibrated confidence.",
            "message": ("Confirm or correct this suggested dish before collecting." if accepted else
                        "This photo may show an unsupported dish or an unclear image. Enter the dish name manually or try another photo."),
        }

    def predict(self, image):
        return self.predict_from_scores(self.score_features(self.encode_images([image]))[0])


In [ ]:
%%writefile run_clip_comparison.py
"""Predeclared calibration / fresh holdout comparison for Choob's CLIP recognizer.

Run in Colab with the existing CLIP model and processor already loaded:
    from recognition import DishRecognizer
    summary, records = run_comparison(model, processor)

This evaluates prompt / candidate / rejection changes, not trained model weights.
The final sample manifest is fixed before inference. The threshold is frozen on
calibration data before any holdout predictions are computed.
"""
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import json
import math
import time

import requests
from PIL import Image

API = "https://datasets-server.huggingface.co"
DATASET = "ethz/food101"
SOURCE = "https://huggingface.co/datasets/ethz/food101"
BASELINE_CANDIDATES = [
    "tonkotsu ramen", "shoyu ramen", "sushi", "dumplings", "fried rice",
    "pad thai", "pizza", "pasta", "hamburger", "tacos",
]
SUPPORTED_GROUPS = [
    ("dumplings", "dumplings"), ("fried_rice", "fried rice"),
    ("hamburger", "hamburger"), ("pad_thai", "pad thai"),
    ("pizza", "pizza"), ("spaghetti_bolognese", "pasta"),
    ("sushi", "sushi"), ("tacos", "tacos"), ("ramen", "ramen"),
]
CALIBRATION_OOD = [
    "apple_pie", "chocolate_cake", "caesar_salad", "french_fries",
    "ice_cream", "steak", "pancakes", "hot_dog",
]
ADDITIONAL_HOLDOUT_OOD = [
    "baklava", "cheesecake", "chicken_wings", "club_sandwich",
    "cup_cakes", "falafel", "nachos", "waffles",
]
MIN_CALIBRATION_RECALL = 0.90


def _api_get(endpoint, **extra):
    """Retry the same preselected request; never replace a failing sample."""
    last_error = None
    for attempt in range(3):
        try:
            response = requests.get(
                API + endpoint,
                params={"dataset": DATASET, "config": "default",
                        "split": "validation", **extra},
                timeout=60,
            )
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as error:
            last_error = error
            if attempt < 2:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Dataset request failed after 3 attempts: {endpoint}") from last_error


def _utc_now():
    return datetime.now(timezone.utc).isoformat()


def _write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def _sample_plan():
    """Offsets within each class block, fixed independently of predictions."""
    plan = []
    for partition, start in [("calibration", 20), ("holdout", 100)]:
        for source_label, expected in SUPPORTED_GROUPS:
            plan.append({
                "partition": partition, "dataset_label": source_label,
                "expected_label": expected,
                "case_type": "ramen_family_only" if expected == "ramen" else "in_scope",
                "ood_group": None, "class_offset": start, "count": 20,
            })
        for source_label in CALIBRATION_OOD:
            plan.append({
                "partition": partition, "dataset_label": source_label,
                "expected_label": None, "case_type": "out_of_scope",
                "ood_group": "calibration_seen_classes", "class_offset": start,
                "count": 10,
            })
    for source_label in ADDITIONAL_HOLDOUT_OOD:
        plan.append({
            "partition": "holdout", "dataset_label": source_label,
            "expected_label": None, "case_type": "out_of_scope",
            "ood_group": "additional_holdout_classes", "class_offset": 100,
            "count": 10,
        })
    return plan


def _download_samples(out):
    image_dir = out / "images"
    image_dir.mkdir(parents=True, exist_ok=True)
    first = _api_get("/first-rows")
    names = next(feature["type"]["names"] for feature in first["features"]
                 if feature["name"] == "label")
    if len(names) != 101 or len(set(names)) != 101:
        raise RuntimeError("Expected the documented Food-101 class schema")
    label_ids = {name: index for index, name in enumerate(names)}

    def inspect_block(block):
        data = _api_get("/rows", offset=block * 250, length=1)
        if len(data["rows"]) != 1:
            raise RuntimeError("Missing class block")
        entry = data["rows"][0]
        if entry["row_idx"] != block * 250:
            raise RuntimeError("Dataset row index differs from requested offset")
        return entry["row"]["label"], entry["row_idx"]

    with ThreadPoolExecutor(max_workers=6) as pool:
        blocks = list(pool.map(inspect_block, range(101)))
    offsets = dict(blocks)
    if len(offsets) != 101 or set(offsets) != set(range(101)):
        raise RuntimeError("Validation rows no longer form 101 distinct 250-row class blocks")

    plan = _sample_plan()
    _write_json(out / "sampling_plan.json", {
        "created_utc": _utc_now(), "dataset": DATASET, "split": "validation",
        "minimum_calibration_recall": MIN_CALIBRATION_RECALL, "groups": plan,
    })

    def select_group(group):
        label_id = label_ids[group["dataset_label"]]
        start = offsets[label_id] + group["class_offset"]
        entries = _api_get("/rows", offset=start, length=group["count"])["rows"]
        expected_rows = list(range(start, start + group["count"]))
        if [entry["row_idx"] for entry in entries] != expected_rows:
            raise RuntimeError("Incomplete or reordered sample group: " + group["dataset_label"])
        if any(entry["row"]["label"] != label_id for entry in entries):
            raise RuntimeError("Unexpected ground truth label: " + group["dataset_label"])
        return [(group, entry) for entry in entries]

    with ThreadPoolExecutor(max_workers=4) as pool:
        selected = [item for group_rows in pool.map(select_group, plan) for item in group_rows]
    selected_ids = [entry["row_idx"] for _, entry in selected]
    if len(selected) != 600 or len(set(selected_ids)) != 600:
        raise RuntimeError("Expected exactly 600 distinct predeclared rows")
    print("Fixed 260 calibration and 340 holdout rows before inference.", flush=True)

    def download(item):
        group, entry = item
        path = image_dir / f"{group['dataset_label']}_{entry['row_idx']}.jpg"
        if path.exists():
            payload = path.read_bytes()
        else:
            last_error = None
            for attempt in range(3):
                try:
                    response = requests.get(entry["row"]["image"]["src"], timeout=60)
                    response.raise_for_status()
                    payload = response.content
                    break
                except requests.RequestException as error:
                    last_error = error
                    if attempt < 2:
                        time.sleep(2 ** attempt)
            else:
                raise RuntimeError(f"Image download failed for row {entry['row_idx']}") from last_error
            path.write_bytes(payload)
        with Image.open(BytesIO(payload)) as original:
            original.verify()
        with Image.open(BytesIO(payload)) as original:
            image = original.convert("RGB")
        record = {
            "dataset": DATASET, "split": "validation", "row_idx": entry["row_idx"],
            **{key: value for key, value in group.items() if key not in {"count", "class_offset"}},
            "class_row_offset": entry["row_idx"] - offsets[label_ids[group["dataset_label"]]],
            "image_file": str(path.relative_to(out)),
            "image_sha256": sha256(payload).hexdigest(), "image_width": image.width,
            "image_height": image.height,
        }
        return image, record

    with ThreadPoolExecutor(max_workers=4) as pool:
        samples = list(pool.map(download, selected))
    hashes = [record["image_sha256"] for _, record in samples]
    if len(set(hashes)) != len(hashes):
        raise RuntimeError("Duplicate image bytes detected; stop rather than substitute holdout samples")
    manifest = [record for _, record in samples]
    _write_json(out / "sample_manifest.json", manifest)
    print("Downloaded and verified every image; all 600 source rows and hashes are distinct.", flush=True)
    return samples


def _select_threshold(calibration_records, minimum_recall=MIN_CALIBRATION_RECALL):
    supported = [record for record in calibration_records if record["case_type"] != "out_of_scope"]
    margins = sorted((float(record["optimized_scores"]["support_margin"])
                      for record in supported), reverse=True)
    if not margins or any(not math.isfinite(value) for value in margins):
        raise ValueError("Calibration margins must be finite and nonempty")
    required = math.ceil(minimum_recall * len(margins))
    threshold = margins[required - 1]
    return threshold, {
        "rule": "Largest support-margin threshold retaining at least 90% of calibration in-scope images; acceptance uses margin >= threshold.",
        "required_supported_accepts": required, "supported_images": len(margins),
        "actual_supported_accepts": sum(value >= threshold for value in margins),
        "minimum_supported_recall": minimum_recall,
        "threshold": threshold,
        "ties": "All margins equal to the threshold are accepted; recall can exceed the target.",
    }


def _canonical_label(label):
    return "ramen" if label in {"tonkotsu ramen", "shoyu ramen"} else label


def _rate(numerator, denominator):
    return numerator / denominator if denominator else None


def _evaluate_records(records, system):
    supported = [row for row in records if row["case_type"] != "out_of_scope"]
    mapped = [row for row in supported if row["case_type"] == "in_scope"]
    ramen = [row for row in supported if row["case_type"] == "ramen_family_only"]
    ood = [row for row in records if row["case_type"] == "out_of_scope"]

    def correct(row, topk=1):
        names = [_canonical_label(item["dish_name"]) for item in row[system]["top_3"][:topk]]
        return row["expected_label"] in names

    accepted = [row for row in records if row[system]["accepted"]]
    supported_accepted = [row for row in supported if row[system]["accepted"]]
    correct_accepted = sum(correct(row) for row in supported_accepted)
    ood_accepted = sum(row[system]["accepted"] for row in ood)
    per_category = {}
    for source_label in sorted({row["dataset_label"] for row in records}):
        rows = [row for row in records if row["dataset_label"] == source_label]
        is_ood = rows[0]["case_type"] == "out_of_scope"
        accepted_rows = [row for row in rows if row[system]["accepted"]]
        per_category[source_label] = {
            "n": len(rows), "accepted": len(accepted_rows), "rejected": len(rows) - len(accepted_rows),
            "top1_correct": None if is_ood else sum(correct(row) for row in rows),
            "top3_correct": None if is_ood else sum(correct(row, 3) for row in rows),
            "correct_and_accepted": None if is_ood else sum(correct(row) for row in accepted_rows),
        }
    ood_groups = {}
    for group in sorted({row["ood_group"] for row in ood}):
        rows = [row for row in ood if row["ood_group"] == group]
        count = sum(row[system]["accepted"] for row in rows)
        ood_groups[group] = {"n": len(rows), "false_accepts": count,
                             "false_accept_rate": _rate(count, len(rows))}
    return {
        "total_images": len(records), "supported_images": len(supported),
        "mapped_non_ramen_images": len(mapped), "ramen_family_images": len(ramen),
        "out_of_scope_images": len(ood),
        "scope_top1_correct": sum(correct(row) for row in supported),
        "scope_top1_accuracy": _rate(sum(correct(row) for row in supported), len(supported)),
        "scope_top3_correct": sum(correct(row, 3) for row in supported),
        "scope_top3_accuracy": _rate(sum(correct(row, 3) for row in supported), len(supported)),
        "mapped_non_ramen_top1_accuracy": _rate(sum(correct(row) for row in mapped), len(mapped)),
        "mapped_non_ramen_top3_accuracy": _rate(sum(correct(row, 3) for row in mapped), len(mapped)),
        "ramen_family_top1_accuracy": _rate(sum(correct(row) for row in ramen), len(ramen)),
        "ramen_family_top3_accuracy": _rate(sum(correct(row, 3) for row in ramen), len(ramen)),
        "supported_accepts": len(supported_accepted),
        "supported_rejections": len(supported) - len(supported_accepted),
        "supported_acceptance_rate": _rate(len(supported_accepted), len(supported)),
        "scope_rejection_rate": _rate(len(supported) - len(supported_accepted), len(supported)),
        "correct_and_accepted": correct_accepted,
        "correct_and_accepted_rate": _rate(correct_accepted, len(supported)),
        "accepted_in_scope_accuracy": _rate(correct_accepted, len(supported_accepted)),
        "all_accepts": len(accepted),
        "accepted_supported_precision": _rate(correct_accepted, len(accepted)),
        "ood_false_accepts": ood_accepted, "ood_false_accept_rate": _rate(ood_accepted, len(ood)),
        "ood_groups": ood_groups, "per_category": per_category,
        "all_cases_decision_accuracy": _rate(correct_accepted + len(ood) - ood_accepted, len(records)),
    }


def run_comparison(model, processor, out_dir="choob_clip_comparison", batch_size=16,
                   result_path=None):
    """Calibrate once, freeze, then evaluate all 340 predeclared holdout photos.

    The output directory retains a sampling plan, complete image manifest,
    frozen calibration record and final JSON. Reusing completed output is
    rejected so an accidental rerun cannot silently overwrite the holdout.
    """
    from recognition import DishRecognizer

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    destination = Path(result_path) if result_path else out / "clip_comparison.json"
    if destination.exists():
        raise FileExistsError("Completed comparison already exists; preserve its frozen holdout result")
    samples = _download_samples(out)
    calibration = [(image, row) for image, row in samples if row["partition"] == "calibration"]
    holdout = [(image, row) for image, row in samples if row["partition"] == "holdout"]
    if len(calibration) != 260 or len(holdout) != 340:
        raise RuntimeError("Unexpected calibration / holdout counts")
    model.eval()
    recognizer = DishRecognizer(model, processor)
    def score_partition(partition_samples):
        started = time.perf_counter()
        image_features = recognizer.encode_images([image for image, _ in partition_samples],
                                                 batch_size=batch_size)
        elapsed = time.perf_counter() - started
        baseline = recognizer.baseline_scores(image_features)
        optimized = recognizer.score_features(image_features)
        if len(baseline) != len(partition_samples) or len(optimized) != len(partition_samples):
            raise RuntimeError("Model output count differs from the fixed sample count")
        records = []
        for (_, source), original, improved in zip(partition_samples, baseline, optimized):
            original = {
                **original, "accepted": True, "status": "supported",
                "dish_name": original["top_3"][0]["dish_name"],
                "rejection_policy": "none_original_forced_choice",
            }
            records.append({**source, "baseline": original, "optimized_scores": improved})
        return records, elapsed

    calibration_records, calibration_encoding_seconds = score_partition(calibration)
    threshold, selection = _select_threshold(calibration_records)
    recognizer.threshold = threshold
    for row in calibration_records:
        row["optimized"] = {**row["optimized_scores"],
                            **recognizer.predict_from_scores(row["optimized_scores"], threshold=threshold)}
    frozen = {
        "frozen_utc": _utc_now(), "selection": selection,
        "calibration_baseline": _evaluate_records(calibration_records, "baseline"),
        "calibration_optimized": _evaluate_records(calibration_records, "optimized"),
        "calibration_records": calibration_records,
        "holdout_prediction_started": False,
    }
    _write_json(out / "frozen_calibration.json", frozen)
    print("CALIBRATION_FROZEN", json.dumps(selection), flush=True)
    holdout_started_utc = _utc_now()
    holdout_records, holdout_encoding_seconds = score_partition(holdout)
    for row in holdout_records:
        row["optimized"] = {**row["optimized_scores"],
                            **recognizer.predict_from_scores(row["optimized_scores"], threshold=threshold)}
    records = calibration_records + holdout_records
    recognition_file = Path(__import__("recognition").__file__)
    summary = {
        "created_utc": _utc_now(), "source_url": SOURCE,
        "model": getattr(model.config, "_name_or_path", "openai/clip-vit-base-patch32"),
        "device": str(next(model.parameters()).device),
        "recognition_source_sha256": sha256(recognition_file.read_bytes()).hexdigest(),
        "total_images": len(records), "calibration_images": len(calibration_records),
        "holdout_images": len(holdout_records), "threshold_selection": selection,
        "threshold_frozen_utc": frozen["frozen_utc"], "holdout_started_utc": holdout_started_utc,
        "baseline_candidate_labels": BASELINE_CANDIDATES,
        "calibration": {"baseline": frozen["calibration_baseline"],
                        "optimized": frozen["calibration_optimized"]},
        "holdout": {"baseline": _evaluate_records(holdout_records, "baseline"),
                    "optimized": _evaluate_records(holdout_records, "optimized")},
        "encoding_timing": {
            "batch_size": batch_size, "calibration_total_seconds": calibration_encoding_seconds,
            "holdout_total_seconds": holdout_encoding_seconds,
            "holdout_seconds_per_image": holdout_encoding_seconds / len(holdout_records),
            "definition": "Shared batched image encoding only; not separate baseline/optimized latency, and excludes downloads, text-feature setup, scoring, Flask and browser upload.",
        },
        "sampling": "Fixed within-class offsets 20..39 calibration and 100..119 holdout for nine supported families; offsets 20..29 and 100..109 for eight calibration-seen out-of-scope classes; 100..109 for eight additional holdout out-of-scope classes. Every row label and image hash verified before inference, with no post-prediction substitutions.",
        "limitations": [
            "An application inference optimization; no CLIP weight fine-tuning was performed.",
            "This predeclared Food-101 subset is not a representative real-world accuracy estimate.",
            "Ramen correctness is family-only; the dataset cannot establish tonkotsu versus shoyu subtype accuracy.",
            "Additional holdout OOD classes have no calibration photos, but their class names may be in the predefined background candidates; this is not unseen-label OOD generalization.",
            "All images are food; non-food, mixed-dish and real user uploads remain untested in this comparison.",
            "Food-101 validation is held out from Food-101 training; CLIP pretraining overlap is unknown.",
            "Relative softmax values are not calibrated correctness probabilities.",
            "The original forced-choice baseline accepts every photo; rejection improves safety at the cost of some supported-food coverage.",
        ],
    }
    destination.parent.mkdir(parents=True, exist_ok=True)
    _write_json(destination, {"summary": summary, "records": records})
    _write_json(out / "summary.json", summary)
    print("COMPARISON_SUMMARY", json.dumps(summary, allow_nan=False), flush=True)
    return summary, records


In [ ]:
import json
from pathlib import Path
from transformers import CLIPModel, CLIPProcessor
from recognition import VERSION, vocabulary_fingerprint
from run_clip_comparison import run_comparison

model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
model.eval()
summary, records = run_comparison(model, processor)
configuration = {'version': VERSION, 'vocabulary_fingerprint': vocabulary_fingerprint(), 'support_margin_threshold': summary['threshold_selection']['threshold'], 'selection': summary['threshold_selection'], 'calibration_images': 260, 'holdout_images': 340}
Path('recognition_config.json').write_text(json.dumps(configuration, indent=2))
print('Saved calibrated configuration. Preserve it with recognition.py.')


In [ ]:
import json
from pathlib import Path
from IPython.display import display, HTML
s = json.loads(Path('choob_clip_comparison/clip_comparison.json').read_text())['summary']
b, v = s['holdout']['baseline'], s['holdout']['optimized']
def fraction(a, n): return f'{a}/{n} ({a/n:.1%})'
rows = [
    ('Supported Top-1 ranking', fraction(b['scope_top1_correct'],180), fraction(v['scope_top1_correct'],180)),
    ('Unsupported photos marked uncertain', fraction(160-b['ood_false_accepts'],160), fraction(160-v['ood_false_accepts'],160)),
    ('Supported photos accepted', fraction(b['supported_accepts'],180), fraction(v['supported_accepts'],180)),
    ('Correct among accepted supported photos', fraction(b['correct_and_accepted'],b['supported_accepts']), fraction(v['correct_and_accepted'],v['supported_accepts'])),
    ('Supported Top-3 ranking', fraction(b['scope_top3_correct'],180), fraction(v['scope_top3_correct'],180)),
]
body = ''.join(f'<tr><td style="padding:7px 5px;border-bottom:1px solid #e2e8ed">{label}</td><td style="padding:7px 5px">{old}</td><td style="padding:7px 5px;font-weight:bold">{new}</td></tr>' for label, old, new in rows)
display(HTML(f'''<div style="font-family:Arial,sans-serif;background:white;color:#17212f;padding:16px;border:1px solid #cbd5df;border-radius:12px;max-width:1000px"><h2 style="font-size:20px;margin:0 0 7px;color:#17212f">Choob Recognition v2 — Fresh Image Test</h2><div style="font-size:12px;color:#52616f;margin-bottom:12px">600 new Food-101 photos · 260 calibration + 340 frozen holdout · CPU</div><table style="border-collapse:collapse;width:100%;font-size:12px;text-align:left"><tr style="background:#e9f3ef"><th style="padding:8px 5px">Holdout measure</th><th>Original</th><th>Optimized</th></tr>{body}</table><p style="font-size:12px;line-height:1.5;margin:10px 0 5px"><b>Tradeoff:</b> 16/180 supported photos require manual entry; 14 had a correct first suggestion. One nachos photo was still accepted as tacos.</p><p style="font-size:11px;line-height:1.4;color:#52616f;margin:0">Same pretrained CLIP weights, improved prompts and scope check. Ramen scored at family level. Relative scores are not confidence. All test images are food; real-world and mixed-dish performance remains unverified. Flask accept/reject and invalid-upload checks passed.</p></div>'''))
